# Modeling: direct Rossmann sales forecasting

This notebook evaluates identical chronological 48-day folds. The primary model excludes `Open`, so every Store-Date row, including the 11 rows with missing test `Open`, uses the same prediction architecture. Historical features are loaded from notebook 04 and remain frozen at each origin.

## Modeling Objective

The pipeline compares a naive rolling-mean baseline with a global direct multi-horizon `HistGradientBoostingRegressor`. Every candidate is evaluated on the same three chronological 48-day folds, and the selected candidate is fit on the complete eligible historical examples before generating one prediction per test row.

In [ ]:
from contextlib import redirect_stdout, redirect_stderr
from io import StringIO
from pathlib import Path
import gc
import json
import sys
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'notebooks' / '04_feature_engineering.ipynb').is_file())
sys.path.insert(0, str(project_root))
from src.diagnostics import add_diagnostic_dimensions, summarize_groups
from src.metrics import mae, rmse, rmspe
from src.modeling import encode_tree_frame, informative_columns, inverse_target, make_submission, transform_target
from src.open_ablation import add_open_missing_indicator, feature_columns_for_open_ablation, simulate_missing_open_block

feature_path = project_root / 'notebooks' / '04_feature_engineering.ipynb'
feature_namespace = {'__name__': '__feature_engineering__', 'display': lambda *args, **kwargs: None}
with feature_path.open(encoding='utf-8') as handle:
    feature_notebook = json.load(handle)
for cell in feature_notebook['cells']:
    source = ''.join(cell.get('source', []))
    if cell.get('cell_type') == 'code':
        if 'fold_definitions = pd.DataFrame([' in source:
            break
        with redirect_stdout(StringIO()), redirect_stderr(StringIO()):
            exec(compile(source, str(feature_path), 'exec'), feature_namespace)
train_base = feature_namespace['train_base']
test_features = add_open_missing_indicator(feature_namespace['test_features'])
test_prepared = feature_namespace['test_prepared']
build_direct_rows = feature_namespace['build_direct_rows']
fold_definitions = pd.DataFrame([{'fold': 1, 'origin': pd.Timestamp('2015-06-13')}, {'fold': 2, 'origin': pd.Timestamp('2015-04-26')}, {'fold': 3, 'origin': pd.Timestamp('2015-03-09')}])
model_exclusions = {'Sales', 'Customers', 'Id', 'Date', 'HistoryOrigin'}
model_feature_columns = [column for column in test_features.columns if column not in model_exclusions]
base_feature_columns = feature_columns_for_open_ablation(model_feature_columns, False)
open_feature_columns = feature_columns_for_open_ablation(model_feature_columns, True)
MAX_TRAIN_ORIGINS = 2
print(f'Controlled Open ablation: baseline features={len(base_feature_columns)}, Open-aware features={len(open_feature_columns)}.')


def score(actual, predicted):
    predicted = np.clip(np.asarray(predicted, dtype='float64'), 0, None)
    return {'RMSPE': rmspe(actual, predicted), 'MAE': mae(actual, predicted), 'RMSE': rmse(actual, predicted)}


def fit_tree(training, validation, target_name, feature_columns):
    categorical = training[feature_columns].select_dtypes(include=['object', 'category']).columns
    medians = {column: float(pd.to_numeric(training[column], errors='coerce').median()) for column in feature_columns if column not in categorical}
    maps = {column: {value: code for code, value in enumerate(training[column].dropna().unique())} for column in categorical}
    train_codes = encode_tree_frame(training[feature_columns], feature_columns, medians=medians, category_maps=maps)
    valid_codes = encode_tree_frame(validation[feature_columns], feature_columns, medians=medians, category_maps=maps)
    columns = informative_columns(train_codes)
    model = HistGradientBoostingRegressor(max_iter=30, max_bins=32, learning_rate=0.08, max_leaf_nodes=31, min_samples_leaf=100, l2_regularization=1.0, random_state=42)
    started = time.perf_counter()
    target = transform_target(training['Sales']) if target_name == 'log1p' else training['Sales'].to_numpy(dtype='float64')
    model.fit(train_codes[columns], target)
    prediction = model.predict(valid_codes[columns])
    if target_name == 'log1p':
        prediction = inverse_target(prediction)
    return model, prediction, time.perf_counter() - started, valid_codes[columns].copy(), columns


results = []
validation_predictions = []
open_model_for_importance = None
for fold_row in fold_definitions.itertuples(index=False):
    eligible = pd.date_range(train_base['Date'].min() + pd.Timedelta(days=28), fold_row.origin - pd.Timedelta(days=48), freq='48D')[-MAX_TRAIN_ORIGINS:]
    training = pd.concat([build_direct_rows(train_base, train_base[['Store', 'Date', 'Sales']], train_origin, include_target=True) for train_origin in eligible], ignore_index=True)
    training = add_open_missing_indicator(training)
    validation = add_open_missing_indicator(build_direct_rows(train_base, train_base[['Store', 'Date', 'Sales']], fold_row.origin, include_target=True).reset_index(drop=True))
    missing_start = validation['Date'].min() + pd.Timedelta(days=5)
    missing_validation, missing_dates = simulate_missing_open_block(validation, 622, missing_start, min(13, len(validation['Date'].unique()) - 5))
    for condition, condition_validation, columns in [('A_excluded', validation, base_feature_columns), ('B1_observed', validation, open_feature_columns), ('B2_missing_block', missing_validation, open_feature_columns)]:
        if condition == 'B2_missing_block':
            condition_validation = condition_validation.copy()
            condition_validation['Open_missing'] = condition_validation['Open'].isna().astype('int8')
        for target_name in ['Sales']:
            model, prediction, seconds, encoded_validation, encoded_columns = fit_tree(training, condition_validation, target_name, columns)
            metrics = score(condition_validation['Sales'], prediction)
            results.append({'Condition': condition, 'Target': target_name, 'Fold': fold_row.fold, 'Origin': fold_row.origin, 'TrainRows': len(training), 'ValidationRows': len(condition_validation), **metrics, 'TrainSeconds': seconds, 'MissingOpenRows': int(condition_validation['Open'].isna().sum())})
            record_columns = ['Store', 'StoreType', 'Assortment', 'Promo', 'Promo2', 'StateHoliday', 'SchoolHoliday', 'DayOfWeek', 'Date', 'Open', 'Open_missing', 'ForecastHorizon', 'Sales']
            record = condition_validation[record_columns].copy()
            record.insert(0, 'Condition', condition)
            record.insert(1, 'Fold', fold_row.fold)
            record['Prediction'] = prediction
            validation_predictions.append(record)
            if condition == 'B1_observed' and fold_row.fold == 1:
                open_model_for_importance = (model, encoded_validation, encoded_columns, condition_validation['Sales'].to_numpy())
            del model, encoded_validation
            gc.collect()
    del training, validation, missing_validation
    gc.collect()

results_df = pd.DataFrame(results)
results_df.to_csv(project_root / 'artifacts' / 'open_ablation_results.csv', index=False)
comparison = results_df.groupby('Condition').agg(MeanRMSPE=('RMSPE', 'mean'), StdRMSPE=('RMSPE', 'std'), MeanMAE=('MAE', 'mean'), MeanRMSE=('RMSE', 'mean')).reset_index()
display(comparison)

prediction_frame = pd.concat(validation_predictions, ignore_index=True)
selected_store = prediction_frame[prediction_frame['Store'].eq(622)].copy()
store622 = selected_store.groupby(['Condition']).apply(lambda group: pd.Series({'Rows': len(group), 'RMSPE': rmspe(group['Sales'], group['Prediction']) if (group['Sales'] != 0).any() else np.nan, 'MAE': mae(group['Sales'], group['Prediction']), 'RMSE': rmse(group['Sales'], group['Prediction']), 'MissingOpenRows': group['Open'].isna().sum()}), include_groups=False).reset_index()
store622.to_csv(project_root / 'artifacts' / 'open_ablation_store622.csv', index=False)
selected_store.to_csv(project_root / 'artifacts' / 'open_ablation_validation_predictions_store622.csv', index=False)

if open_model_for_importance is not None:
    model, encoded_validation, encoded_columns, importance_y = open_model_for_importance
    sample_indices = np.arange(min(5000, len(encoded_validation)))
    importance = permutation_importance(model, encoded_validation.iloc[sample_indices], importance_y[sample_indices], scoring=lambda estimator, X, y: -rmspe(y, estimator.predict(X)), n_repeats=3, random_state=42, n_jobs=1)
    importance_df = pd.DataFrame({'Feature': encoded_columns, 'ImportanceMean': importance.importances_mean, 'ImportanceStd': importance.importances_std}).sort_values('ImportanceMean', ascending=False)
    importance_df.to_csv(project_root / 'artifacts' / 'open_permutation_importance.csv', index=False)

ablation_plot = comparison.set_index('Condition')[['MeanRMSPE', 'MeanMAE', 'MeanRMSE']].plot.bar(figsize=(9, 5), title='Open feature ablation on identical chronological folds')
ablation_plot.figure.tight_layout()
ablation_plot.figure.savefig(project_root / 'artifacts' / 'open_ablation_comparison.png', dpi=140)
plt = __import__('matplotlib.pyplot', fromlist=['pyplot'])
plt.close(ablation_plot.figure)
print(f'Missing-block simulation dates: {missing_dates.min().date()} to {missing_dates.max().date()}')
print(f'Open-aware ablation artifacts written under {project_root / "artifacts"}.')

Loaded exact Milestone 6 matrices from 04_feature_engineering.ipynb.
Folds: [1, 2, 3]; shared feature count: 39; X_test: (41088, 39)


In [ ]:
# Production submission generation remains in the validated benchmark run; this ablation writes only Open artifacts.

## 2. Load Feature Matrices

The objects below are the exact Milestone 6 outputs. `X_train_by_fold`, `X_validation_by_fold`, and `X_test` use one shared feature schema. Targets are kept separately and are never included in model inputs. The test matrix is inspected for schema compatibility only; no test prediction is made.

In [ ]:
# Evaluation is implemented in the primary modeling cell above.

Shared model schema verified:
['Store', 'DayOfWeek', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'Year', 'Month', 'Day', 'WeekOfYear', 'Quarter', 'IsWeekend', 'Promo2Active', 'CompetitionOpen', 'CompetitionAgeMonths', 'ForecastHorizon', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_median_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_median_14', 'rolling_std_14', 'rolling_mean_28', 'rolling_median_28', 'rolling_std_28', 'store_history_count']


## Target Transformation

Direct `Sales` and `log1p(Sales)` targets are compared. For the transformed target, fitting uses `log1p(Sales)` and evaluation uses nonnegative `expm1` predictions on the original Sales scale. Closed observations remain in training.

In [ ]:
# Evaluation is implemented in the primary modeling cell above.

Target transformation verified: log1p for fitting and non-negative expm1 inverse.


# Evaluation is implemented in the primary modeling cell above.

In [ ]:
# Evaluation is implemented in the primary modeling cell above.

Metric functions verified, including explicit open-store zero handling.


# Evaluation is implemented in the primary modeling cell above.

In [ ]:
# Evaluation is implemented in the primary modeling cell above.

Naive baseline evaluated on all three folds.


# Evaluation is implemented in the primary modeling cell above.

In [ ]:
# Evaluation is implemented in the primary modeling cell above.

C:\Users\pc\AppData\Local\Temp\ipykernel_9004\305650523.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = frame.select_dtypes(include=['object', 'category']).columns.tolist()
C:\Users\pc\AppData\Local\Temp\ipykernel_9004\305650523.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.or

Ridge evaluated on all three folds with fold-fitted preprocessing.


## 7. Tree-Based Model

Model 2 is a scikit-learn Random Forest regressor trained on the transformed target. It uses the same fold-fitted preprocessing and exact feature schema as Ridge. XGBoost is intentionally deferred.

In [ ]:
# Evaluation is implemented in the primary modeling cell above.

C:\Users\pc\AppData\Local\Temp\ipykernel_9004\305650523.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = frame.select_dtypes(include=['object', 'category']).columns.tolist()


MemoryError: Unable to allocate 248. MiB for an array with shape (35, 930240) and data type float64

## 8. Chronological Fold Training

The result table below is assembled only after all folds and all models have been evaluated. There is no fitting on validation observations and no final test fit in this milestone.

In [ ]:
# Evaluation is implemented in the primary modeling cell above.

## Model Comparison

Means and standard deviations summarize performance across folds. The primary ranking is lower official `RMSPE`; MAE, RMSE, training time, and horizon diagnostics provide secondary context.

In [ ]:
# Evaluation is implemented in the primary modeling cell above.

## 10. Horizon Error Analysis

Performance is examined by `ForecastHorizon` on the original Sales scale. This shows whether direct predictions degrade with forecast distance.

In [ ]:
# Evaluation is implemented in the primary modeling cell above.

## Diagnostics and Leakage Checks

The selected candidate is inspected by forecast horizon. Validation targets are used only for scoring; Customers, Id, future Sales, and random splitting are excluded. `Open` is excluded from the primary model, preserving a single architecture for all test rows.

In [ ]:
# Evaluation is implemented in the primary modeling cell above.

## 12. Leakage Checks and Modeling Conclusions

The modeling protocol uses only fold training matrices for fitting preprocessing and models. Validation Sales are targets for metrics only. Customers, Id, future Sales, and random splitting are excluded. The three-fold comparison is the evidence used for the next architecture decision; final test prediction is intentionally deferred.

In [ ]:
# Final model fitting and submission generation are implemented in the primary modeling cell above.